In [ ]:
# -*- coding: utf-8 -*-
"""
LoRA Baseline - Spike Detection

Standard LoRA implementation for comparison with CDWF.

Tests multiple LoRA ranks: 1, 2, 4, 8, 16
"""

import os
import gc
import math
import random
import time
import copy
import re

from tqdm import tqdm
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import OneCycleLR

from sklearn.metrics import roc_auc_score


# ================================================================
# GPU POWER MONITORING
# ================================================================
try:
    import pynvml

    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print("GPU power monitoring enabled")
except Exception:
    NVML_AVAILABLE = False
    _nvml_handle = None
    print("GPU power monitoring unavailable")


def read_gpu_power_w():
    if not NVML_AVAILABLE:
        return 0.0
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_nvml_handle) / 1000.0
    except Exception:
        return 0.0


# ================================================================
# CONFIGURATION
# ================================================================
EXPERIMENT_NAME = "spike_resnet50_lora_baseline"
SEED = 42

# Data paths
NORMAL_DIR = "/normal_vs_spike/data/normal"
ATTACK_DIR = "/normal_vs_spike/data/attack"
BIAS_CKPT = "best_resnet1d_basicblock_bias.pth"

# Model
BASE_CHANNELS = 64
USE_MAXPOOL = False

# Training
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 10
MAX_LR = 3e-4
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0

# LoRA configuration
LORA_RANKS = [1, 2, 4, 8, 16]
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# Data
NUM_CLASSES = 2
FULLFT_CSV = "spike_resnet50_fullft_results.csv"
RESULT_CSV = "spike_resnet50_lora_baseline_results.csv"

print(f"\n{'=' * 70}")
print("  LORA BASELINE - SPIKE DETECTION")
print(f"{'=' * 70}")
print("Model: ResNet1D with BasicBlock + GroupNorm")
print(f"Testing LoRA ranks: {LORA_RANKS}")
print(f"LoRA alpha: {LORA_ALPHA}")
print(f"Training epochs: {EPOCHS}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"{'=' * 70}\n")


# ================================================================
# REPRODUCIBILITY
# ================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")


# ================================================================
# RESNET1D WITH MASKED GLOBAL AVERAGE POOLING
# ================================================================
def pick_gn_groups(channels):
    for groups in [32, 16, 8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class BasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        groups = pick_gn_groups(planes)

        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(groups, planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(groups, planes)

        self.down = None
        if stride != 1 or in_planes != planes:
            self.down = nn.Sequential(
                nn.Conv1d(in_planes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(pick_gn_groups(planes), planes),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(x)
        out = self.relu(out + identity)
        return out


class ResNet1D(nn.Module):
    def __init__(self, base=64, use_maxpool=False, num_classes=2):
        super().__init__()
        self.use_maxpool = use_maxpool

        self.conv1 = nn.Conv1d(1, base, 7, stride=2, padding=3, bias=False)
        self.gn1 = nn.GroupNorm(pick_gn_groups(base), base)
        self.relu = nn.ReLU(inplace=True)

        if use_maxpool:
            self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)

        self.inplanes = base
        self.layer1 = self._make_layer(base, blocks=2, stride=1)
        self.layer2 = self._make_layer(base * 2, blocks=2, stride=2)
        self.layer3 = self._make_layer(base * 4, blocks=2, stride=2)
        self.layer4 = self._make_layer(base * 8, blocks=2, stride=2)

        self.fc = nn.Linear(base * 8, num_classes)

        for module in self.modules():
            if isinstance(module, nn.Conv1d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.GroupNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def _make_layer(self, planes, blocks, stride):
        layers = [BasicBlock1D(self.inplanes, planes, stride)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.inplanes, planes, 1))
        return nn.Sequential(*layers)

    def forward(self, x, mask=None):
        x = self.relu(self.gn1(self.conv1(x)))
        if self.use_maxpool:
            x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        if mask is not None:
            pooled_mask = F.adaptive_max_pool1d(mask, x.size(-1))
            weighted_sum = (x * pooled_mask).sum(dim=-1)
            denom = pooled_mask.sum(dim=-1).clamp_min(1.0)
            x = weighted_sum / denom
        else:
            x = F.adaptive_avg_pool1d(x, 1).squeeze(-1)

        x = self.fc(x)
        return x


# ================================================================
# STANDARD LORA IMPLEMENTATION
# ================================================================
class LoRALayer(nn.Module):
    """Standard LoRA wrapper for Conv1d."""

    def __init__(self, base_layer, rank=4, alpha=16, dropout=0.0):
        super().__init__()
        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha

        for param in self.base_layer.parameters():
            param.requires_grad = False

        if isinstance(base_layer, nn.Conv1d):
            in_channels = base_layer.in_channels
            out_channels = base_layer.out_channels
            kernel_size = base_layer.kernel_size
            stride = base_layer.stride
            padding = base_layer.padding

            self.lora_A = nn.Conv1d(
                in_channels,
                rank,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                bias=False,
            )
            self.lora_B = nn.Conv1d(
                rank,
                out_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False,
            )

            nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B.weight)

            self.dropout = nn.Dropout(dropout) if dropout > 0 else None
            self.scaling = alpha / rank

    def forward(self, x):
        base_out = self.base_layer(x)

        lora_out = self.lora_A(x)
        if self.dropout is not None:
            lora_out = self.dropout(lora_out)
        lora_out = self.lora_B(lora_out)

        return base_out + self.scaling * lora_out


def add_lora_to_model(model, rank=4, alpha=16, dropout=0.0):
    """Add LoRA to conv2 layers in residual blocks."""
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)
        for block in layer:
            if hasattr(block, "conv2"):
                block.conv2 = LoRALayer(block.conv2, rank, alpha, dropout)
    return model


# ================================================================
# MODEL UTILITIES
# ================================================================
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable


def freeze_base_model(model):
    """Freeze all parameters except LoRA layers and classifier."""
    for name, param in model.named_parameters():
        if "lora" not in name and not name.startswith("fc"):
            param.requires_grad = False


# ================================================================
# DATA LOADING
# ================================================================
class SpikeDataset(Dataset):
    def __init__(self, X, M, y):
        self.X = X.astype(np.float32)
        self.M = M.astype(np.float32)
        self.y = y.astype(np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        vals = self.X[idx]
        msk = self.M[idx]

        valid = msk > 0.5
        if valid.any():
            valid_vals = vals[valid]
            mu = float(valid_vals.mean())
            sd = float(valid_vals.std() + 1e-6)
            vals = (vals - mu) / sd
        else:
            vals = vals * 0.0

        vals = torch.from_numpy(vals[None, :])
        msk = torch.from_numpy(msk[None, :])
        lab = torch.tensor(self.y[idx], dtype=torch.long)
        return vals, msk, lab


def load_spike_data(normal_dir, attack_dir):
    """Load spike data with padding and masking."""
    re_normal = re.compile(r"^n_(\d+)\.npy$", re.IGNORECASE)
    re_attack = re.compile(r"^a_(\d+)\.npy$", re.IGNORECASE)

    def map_indices(dir_path, pattern):
        mapping = {}
        for fn in os.listdir(dir_path):
            match = pattern.match(fn)
            if match:
                mapping[int(match.group(1))] = fn
        return mapping

    def safe_load_array(path):
        try:
            arr = np.load(path, allow_pickle=False)
            return np.asarray(arr).ravel().astype(np.float32)
        except Exception as exc:
            print(f"[load warning] {path} -> {exc}")
            return None

    n_map = map_indices(normal_dir, re_normal)
    a_map = map_indices(attack_dir, re_attack)
    common_idx = sorted(set(n_map.keys()) & set(a_map.keys()))

    print(f"  Matched indices: {len(common_idx)}")

    samples, labels, pair_ids = [], [], []

    for idx in tqdm(common_idx, desc="Loading"):
        n_path = os.path.join(normal_dir, n_map[idx])
        a_path = os.path.join(attack_dir, a_map[idx])

        n_arr = safe_load_array(n_path)
        a_arr = safe_load_array(a_path)
        if n_arr is None or a_arr is None:
            continue

        samples.append(n_arr)
        labels.append(0)
        pair_ids.append(idx)

        samples.append(a_arr)
        labels.append(1)
        pair_ids.append(idx)

    print(f"  Total samples: {len(labels)}")

    lengths = [len(s) for s in samples]
    max_len = max(lengths)

    X_full = np.full((len(samples), max_len), np.nan, np.float32)
    for i, s in enumerate(samples):
        X_full[i, :len(s)] = s

    y_full = np.asarray(labels, dtype=np.int64)
    pair_ids = np.asarray(pair_ids, dtype=np.int64)

    mask_full = ~np.isnan(X_full)
    X_filled = np.where(mask_full, X_full, 0.0).astype(np.float32)
    mask_full = mask_full.astype(np.float32)

    return X_filled, mask_full, y_full, pair_ids


print("Loading spike detection data...")
X_filled, mask_full, y_full, pair_ids = load_spike_data(NORMAL_DIR, ATTACK_DIR)

TRAIN_FRACTION = 0.80
VAL_FRACTION_WITHIN_TRAIN = 0.20

unique_pairs = sorted(set(pair_ids.tolist()))
num_pairs = len(unique_pairs)

rng = np.random.RandomState(SEED)
perm = rng.permutation(num_pairs)
ordered_pairs = [unique_pairs[i] for i in perm]

num_train_pairs_total = int(math.floor(num_pairs * TRAIN_FRACTION))
num_val_pairs = int(math.floor(num_train_pairs_total * VAL_FRACTION_WITHIN_TRAIN))

train_pairs_all = ordered_pairs[:num_train_pairs_total]
test_pairs = ordered_pairs[num_train_pairs_total:]

if num_val_pairs > 0:
    val_pairs = train_pairs_all[:num_val_pairs]
    train_pairs = train_pairs_all[num_val_pairs:]
else:
    val_pairs = []
    train_pairs = train_pairs_all

train_mask = np.isin(pair_ids, train_pairs)
val_mask = np.isin(pair_ids, val_pairs)
test_mask = np.isin(pair_ids, test_pairs)

X_train, M_train, y_train = X_filled[train_mask], mask_full[train_mask], y_full[train_mask]
X_val, M_val, y_val = X_filled[val_mask], mask_full[val_mask], y_full[val_mask]
X_test, M_test, y_test = X_filled[test_mask], mask_full[test_mask], y_full[test_mask]

print(f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}\n")

train_dataset = SpikeDataset(X_train, M_train, y_train)
val_dataset = SpikeDataset(X_val, M_val, y_val)
test_dataset = SpikeDataset(X_test, M_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ================================================================
# TRAINING FUNCTIONS
# ================================================================
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, scheduler, device, power_log=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for vals, masks, targets in tqdm(loader, desc="Train", leave=False):
        if power_log is not None:
            now = time.time()
            power_log["energy_J"] += read_gpu_power_w() * (now - power_log["last_t"])
            power_log["last_t"] = now

        vals, masks, targets = vals.to(device), masks.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(vals, masks)
        loss = criterion(outputs, targets)
        loss.backward()

        if CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)

        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    for vals, masks, targets in tqdm(loader, desc="Eval", leave=False):
        vals, masks, targets = vals.to(device), masks.to(device), targets.to(device)
        outputs = model(vals, masks)
        loss = criterion(outputs, targets)

        probs = torch.softmax(outputs, dim=1)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        all_labels.extend(targets.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    acc = 100.0 * correct / total
    auc = roc_auc_score(all_labels, all_probs) * 100.0

    return total_loss / len(loader), acc, auc


# ================================================================
# LOAD SPIKE FULL_FT BASELINE FROM CSV
# ================================================================
print("=" * 70)
print("  LOADING SPIKE FULL_FT BASELINE")
print("=" * 70 + "\n")

if os.path.exists(FULLFT_CSV):
    print(f"Loading baseline from: {FULLFT_CSV}")
    fullft_df = pd.read_csv(FULLFT_CSV)
    fullft_auc = fullft_df["test_auc"].iloc[0]
    fullft_time = fullft_df["training_time_s"].iloc[0]
    fullft_energy = fullft_df["energy_J"].iloc[0]
    fullft_test_acc = fullft_df["test_acc"].iloc[0]

    print("Loaded Spike Full_FT baseline")
    print(f"  Test Accuracy: {fullft_test_acc:.2f}%")
    print(f"  Test AUC: {fullft_auc:.2f}%")
    print(f"  Training Time: {fullft_time:.1f}s")
    print(f"  Energy: {fullft_energy / 1000:.2f}kJ\n")
else:
    print("Error: Spike full fine-tuning baseline not found.")
    print(f"Expected file: {FULLFT_CSV}")
    print("Run the spike full fine-tuning script first.\n")
    raise SystemExit(1)


# ================================================================
# TRAIN LORA WITH DIFFERENT RANKS
# ================================================================
all_results = []

for lora_rank in LORA_RANKS:
    print("=" * 70)
    print(f"  LORA RANK {lora_rank}")
    print("=" * 70 + "\n")

    torch.cuda.empty_cache()
    gc.collect()
    set_seed(SEED)

    print(f"Creating ResNet1D with LoRA rank-{lora_rank}...")
    model = ResNet1D(base=BASE_CHANNELS, use_maxpool=USE_MAXPOOL, num_classes=NUM_CLASSES)

    if os.path.exists(BIAS_CKPT):
        try:
            state = torch.load(BIAS_CKPT, map_location="cpu")
            model.load_state_dict(state, strict=False)
            print(f"Loaded bias checkpoint from {BIAS_CKPT}")
        except Exception as exc:
            print(f"Could not load checkpoint: {exc}")
            print("Training from scratch")
    else:
        print(f"Bias checkpoint not found: {BIAS_CKPT}")
        print("Training from scratch")

    model = add_lora_to_model(
        model,
        rank=lora_rank,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT,
    )
    freeze_base_model(model)
    model = model.to(device)

    total_params, trainable_params, frozen_params = count_params(model)
    classifier_params = sum(p.numel() for p in model.fc.parameters())
    lora_params = trainable_params - classifier_params

    print("[Model Summary]")
    print(f"  Total params: {total_params:,} ({total_params / 1e6:.2f}M)")
    print(f"  Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"  LoRA params: {lora_params:,}")
    print(f"  Reduction: {total_params / trainable_params:.1f}x\n")

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable, lr=MAX_LR, weight_decay=WEIGHT_DECAY)
    scheduler = OneCycleLR(
        optimizer,
        max_lr=MAX_LR,
        steps_per_epoch=len(train_loader),
        epochs=EPOCHS,
        pct_start=0.15,
        div_factor=10,
        final_div_factor=100,
    )

    print(f"Training for {EPOCHS} epochs...\n")

    start_time = time.time()
    power_log = {"energy_J": 0.0, "last_t": start_time}

    best_val_auc = 0.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            power_log=power_log,
        )
        val_loss, val_acc, val_auc = evaluate(model, val_loader, device)

        print(
            f"Epoch {epoch}/{EPOCHS}: "
            f"train_acc={train_acc:.2f}% val_acc={val_acc:.2f}% val_auc={val_auc:.2f}%"
        )

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())

    training_time = time.time() - start_time
    training_energy = power_log["energy_J"]

    if best_state is not None:
        model.load_state_dict(best_state)

    _, test_acc, test_auc = evaluate(model, test_loader, device)

    retention = test_auc / fullft_auc
    param_reduction = total_params / trainable_params
    time_speedup = fullft_time / training_time
    energy_reduction = fullft_energy / training_energy

    print(f"\n{'=' * 70}")
    print(f"  RESULTS: LORA RANK {lora_rank}")
    print(f"{'=' * 70}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Test AUC: {test_auc:.2f}%")
    print(f"Retention: {retention * 100:.1f}%")
    print(f"Trainable Params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"Parameter Reduction: {param_reduction:.1f}x\n")
    print(f"Training Time: {training_time:.1f}s ({training_time / 60:.2f} min)")
    print(f"Time Speedup: {time_speedup:.2f}x")
    print(f"Energy: {training_energy / 1000:.2f} kJ")
    print(f"Energy Reduction: {energy_reduction:.2f}x")
    print(f"{'=' * 70}\n")

    results = {
        "method": "LoRA",
        "dataset": "spike",
        "lora_rank": lora_rank,
        "lora_alpha": LORA_ALPHA,
        "fullft_test_acc": fullft_test_acc,
        "fullft_auc": fullft_auc,
        "test_acc": test_acc,
        "test_auc": test_auc,
        "retention": retention,
        "trainable_params": trainable_params,
        "trainable_percent": 100 * trainable_params / total_params,
        "param_reduction": param_reduction,
        "training_time_s": training_time,
        "time_speedup": time_speedup,
        "energy_J": training_energy,
        "energy_kJ": training_energy / 1000,
        "energy_reduction": energy_reduction,
    }

    all_results.append(results)


# ================================================================
# SAVE ALL RESULTS
# ================================================================
print("=" * 70)
print("  SUMMARY: ALL LORA CONFIGURATIONS")
print("=" * 70 + "\n")

df = pd.DataFrame(all_results)
print(df.to_string(index=False))
print()

if os.path.exists(RESULT_CSV):
    existing_df = pd.read_csv(RESULT_CSV)
    df = pd.concat([existing_df, df], ignore_index=True)

df.to_csv(RESULT_CSV, index=False)
print(f"Results saved to {RESULT_CSV}\n")

print("=" * 70)
print("  COMPARISON: SPIKE FULL_FT VS LORA")
print("=" * 70)
print(f"{'Rank':<8} {'Retention':<12} {'Speedup':<10} {'Energy':<10} {'Params':<10}")
print("-" * 70)
print(f"{'Full_FT':<8} {'100.0%':<12} {'1.00x':<10} {'1.00x':<10} {'100.0%':<10}")
for result in all_results:
    print(
        f"r={result['lora_rank']:<5} {result['retention'] * 100:>6.1f}%     "
        f"{result['time_speedup']:>6.2f}x    {result['energy_reduction']:>6.2f}x    "
        f"{result['trainable_percent']:>6.2f}%"
    )
print("=" * 70 + "\n")

print("LoRA Baseline Complete")
print(f"Tested ranks: {LORA_RANKS}")
print(f"Best retention: {max(r['retention'] for r in all_results) * 100:.1f}%")
print(f"Best speedup: {max(r['time_speedup'] for r in all_results):.2f}x")